# 02 - Generate images (FLUX.1-dev)

Turns the 1,500 prompts in `data/prompts.json` into the **undegraded** synthetic image set
under `data/synthetic_clean/{class}/`. CCTV degradation is NOT done here - it happens in
step 03 (this keeps the with/without-degradation ablation valid), so these come out as clean photos.

**Runs on a CUDA GPU** (the university VSCode container, or any GPU box) - not from the authoring
machine. Pull the repo there, set up HuggingFace access, then run top-to-bottom.

**Prerequisites (one-time):**
1. Free HuggingFace account.
2. Accept the FLUX.1-dev license: https://huggingface.co/black-forest-labs/FLUX.1-dev (gated).
3. Create a *Read* access token, then on the container either run `huggingface-cli login` or
   `export HF_TOKEN=hf_...` (never commit the token).
4. ~30 GB free disk (FLUX-dev weights ~24 GB + images).

The full run is **resumable** (skips images already on disk) and **reproducible** (fixed seeds,
recorded in `data/synthetic_clean/manifest.csv`).

## 0. Install dependencies (run once per fresh environment)

In [ ]:
# torch (a CUDA build) is assumed already installed on the GPU container.
%pip install -q diffusers transformers accelerate sentencepiece protobuf

In [ ]:
import os, time, csv
from pathlib import Path
import torch
from PIL import Image
from huggingface_hub import login

# Authenticate to HuggingFace for the gated FLUX.1-dev weights.
# Preferred: set HF_TOKEN in the container environment (never commit it).
_tok = os.environ.get('HF_TOKEN')
if _tok:
    login(token=_tok)
    print('Logged in to HuggingFace via HF_TOKEN.')
else:
    print('HF_TOKEN not set - assuming `huggingface-cli login` was already run (cached credentials).')

In [ ]:
import sys

# Make code/ importable whether launched from the repo root or from code/.
HERE = Path.cwd()
CODE_DIR = HERE if HERE.name == 'code' else HERE / 'code'
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
import utils

prompts_by_class = utils.load_prompts()
print('classes:', utils.CLASS_NAMES)
print('prompts per class:', {c: len(prompts_by_class[c]) for c in utils.CLASS_NAMES})
print('output dir:', utils.CLEAN_DIR)

In [ ]:
MODEL_ID = 'black-forest-labs/FLUX.1-dev'

IMAGES_PER_PROMPT = 1   # 1 -> 1500 images (300/class). Raise to 2-3 for a larger set.
GEN_SIZE = 1024         # FLUX native resolution
SAVE_SIZE = 512         # saved undegraded size (step 03 downscales much further anyway)
STEPS = 28              # FLUX.1-dev: ~28-30 steps
GUIDANCE = 3.5          # FLUX.1-dev distilled guidance
PILOT_PER_CLASS = 10

In [ ]:
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
pipe = pipe.to('cuda')
# If VRAM is tight, replace the line above with: pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)
print('Loaded', MODEL_ID)

In [ ]:
def generate(prompt, seed):
    """Generate one GEN_SIZE image for a prompt with a fixed seed (reproducible)."""
    gen = torch.Generator('cpu').manual_seed(seed)
    out = pipe(
        prompt,
        height=GEN_SIZE, width=GEN_SIZE,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
        max_sequence_length=512,
        generator=gen,
    )
    return out.images[0]


def save_image(img, path):
    """Resize to SAVE_SIZE and save as JPEG."""
    img.resize((SAVE_SIZE, SAVE_SIZE), Image.LANCZOS).save(path, quality=95)

## Pilot (10/class) - QA before the full run

Generate 10 images per class into `data/_pilot/` and eyeball the grid. Confirm:
`clean` vs `empty` are distinguishable, `full`/`finished_leftovers` amounts read correctly,
**there is no timestamp / camera / text overlay**, and a single plate fills the frame.
Only scale to the full run once this looks right.

In [ ]:
# Pilot uses the first PILOT_PER_CLASS prompts of each class. Distinct seed range from the full run.
pilot_base = utils.SEED + 90000
n = 0
for cls in utils.CLASS_NAMES:
    d = utils.class_dir(utils.PILOT_DIR, cls)
    for i in range(PILOT_PER_CLASS):
        fp = d / f'{cls}_{i:02d}.jpg'
        if fp.exists():
            continue
        save_image(generate(prompts_by_class[cls][i], pilot_base + n), fp)
        n += 1
print('pilot images generated this run:', n)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(utils.CLASS_NAMES), PILOT_PER_CLASS,
                         figsize=(PILOT_PER_CLASS * 1.5, len(utils.CLASS_NAMES) * 1.7))
for r, cls in enumerate(utils.CLASS_NAMES):
    files = sorted((utils.PILOT_DIR / cls).glob('*.jpg'))
    for c in range(PILOT_PER_CLASS):
        ax = axes[r][c]
        ax.set_xticks([]); ax.set_yticks([])
        if c < len(files):
            ax.imshow(Image.open(files[c]))
        if c == 0:
            ax.set_ylabel(cls, rotation=0, ha='right', va='center', fontsize=9)
plt.suptitle('Pilot QA: clean vs empty distinct? amounts right? NO text/HUD overlay? single plate?', fontsize=10)
plt.tight_layout()
plt.show()

## Full run

Resumable (skips images already on disk) and reproducible (deterministic per-image seeds).
FLUX-dev is slow (~seconds/image); 1,500 images can take ~1-4 h depending on the GPU. Safe to
re-run if interrupted.

In [ ]:
t0 = time.time()
gidx = 0
made = 0
for cls in utils.CLASS_NAMES:
    out_dir = utils.class_dir(utils.CLEAN_DIR, cls)
    for p_i, prompt in enumerate(prompts_by_class[cls]):
        for k in range(IMAGES_PER_PROMPT):
            seed = utils.SEED + gidx
            gidx += 1
            fp = out_dir / f'{cls}_{p_i:04d}_{k}.jpg'
            if fp.exists():
                continue
            save_image(generate(prompt, seed), fp)
            made += 1
            if made % 50 == 0:
                print(f'  {made} new images | {time.time() - t0:.0f}s elapsed')
print('full run done. new images this run:', made, '| elapsed:', f'{time.time() - t0:.0f}s')

In [ ]:
# Build manifest.csv deterministically from the same loop order (robust to resumes).
utils.CLEAN_DIR.mkdir(parents=True, exist_ok=True)
gidx = 0
n = 0
with open(utils.MANIFEST_PATH, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['filepath', 'class', 'seed', 'model', 'prompt'])
    for cls in utils.CLASS_NAMES:
        for p_i, prompt in enumerate(prompts_by_class[cls]):
            for k in range(IMAGES_PER_PROMPT):
                seed = utils.SEED + gidx
                gidx += 1
                fp = utils.CLEAN_DIR / cls / f'{cls}_{p_i:04d}_{k}.jpg'
                if fp.exists():
                    w.writerow([str(fp.relative_to(utils.ROOT_DIR)), cls, seed, MODEL_ID, prompt])
                    n += 1
print('manifest rows:', n, '->', utils.MANIFEST_PATH)

In [ ]:
import random

counts = {cls: len(list((utils.CLEAN_DIR / cls).glob('*.jpg'))) for cls in utils.CLASS_NAMES}
print('images per class:', counts, '| total:', sum(counts.values()))

# Random sample grid across classes (sanity that files are readable).
rng = random.Random(utils.SEED)
fig, axes = plt.subplots(len(utils.CLASS_NAMES), 4, figsize=(4 * 1.6, len(utils.CLASS_NAMES) * 1.7))
for r, cls in enumerate(utils.CLASS_NAMES):
    files = list((utils.CLEAN_DIR / cls).glob('*.jpg'))
    pick = rng.sample(files, min(4, len(files)))
    for c in range(4):
        ax = axes[r][c]
        ax.set_xticks([]); ax.set_yticks([])
        if c < len(pick):
            ax.imshow(Image.open(pick[c]))
        if c == 0:
            ax.set_ylabel(cls, rotation=0, ha='right', va='center', fontsize=9)
plt.tight_layout()
plt.show()